# Milvus的基本使用

# 1、DDL操作

## 1.1 数据库相关操作

### ① 查看数据库

举例1：操作客户端

In [1]:

from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

举例2：列出所有数据库

In [9]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default
rag_demo


### ② 创建数据库

In [8]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name=db_name)

### ③ 删除数据库

如果数据库下有Collection则无法删除，需要先删除它的所有Collection才能删除Database

In [7]:

client.drop_database(db_name = db_name)

## 1.2 Collection相关操作

### ① 切换数据库

In [10]:
client.use_database(db_name=db_name)

### ② 查看数据库下的collections

In [29]:
collections = client.list_collections()

for coll in collections:
    print(coll)

docs


### ③ 创建collection

In [15]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ④ 删除collection

In [24]:
client.drop_collection(collection_name=collection_name)

# 2、DML操作

## 2.1 嵌入模型的初始化

In [26]:
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 初始化嵌入模型
embed_model = init_embeddings(
	model="openai:Pro/BAAI/bge-m3",
	api_key=os.getenv("SILICONFLOW_API_KEY"),
	base_url=os.getenv("SILICONFLOW_BASE_URL"),
)

## 2.2 准备collection

### ① 创建collection

In [28]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ② 查看collection元数据

In [30]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467040847074183555,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467041742317420562,
    'update_timestamp': 467041742317420562
}

## 2.3 准备数据

### ① 准备原始数据

In [32]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

### ② 生成嵌入向量

In [33]:
vectors = embed_model.embed_documents(texts)

### ③ 查看生成的嵌入向量

In [34]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
1024
[-0.011260323226451874, 0.04925568029284477, 0.04267069697380066, -0.0021236573811620474, 0.005432611797004938]


### ④ 封装为可以插入的数据格式

In [35]:
data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

## 2.4 写入数据

### ① 插入数据

In [36]:
insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### ② 手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [37]:
client.flush(collection_name=collection_name)

### ③ 查看collection统计信息

In [38]:
stats = client.get_collection_stats(collection_name=collection_name)

print("stats : ",stats)

stats :  {'row_count': 4}


# 3、DQL操作

## 3.1 扫描数据

In [46]:

iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")

        i += 1

iterator.close()

第1条数据：
id : 0,vector = [-0.011260323226451874, 0.04925568029284477, 0.04267069697380066, -0.0021236573811620474, 0.005432611797004938],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.02432798594236374, 0.0016291062347590923, 0.01846320368349552, 0.00872476864606142, -0.009267804212868214],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.004508086945861578, 0.030777014791965485, 0.0059316931292414665, -0.03660701960325241, -0.0033217482268810272],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第4条数据：
id : 3,vector = [0.005521129816770554, 0.013496095314621925, -0.013929124921560287, -0.02554875798523426, -0.010753572918474674],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


## 3.2 通过主键查询数据

In [50]:

res = client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


3
第1条数据：
id : 0,vector = [-0.011260323226451874, 0.04925568029284477, 0.04267069697380066, -0.0021236573811620474, 0.005432611797004938],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.02432798594236374, 0.0016291062347590923, 0.01846320368349552, 0.00872476864606142, -0.009267804212868214],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.004508086945861578, 0.030777014791965485, 0.0059316931292414665, -0.03660701960325241, -0.0033217482268810272],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


## 3.3 相似度检索

### ① 准备查询嵌入

In [52]:
# 相似度检索
query = "什么是向量数据库？"
query_vector = embed_model.embed_query(query)

### ② 检索

In [55]:
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]:
    print(res)

{'id': 0, 'distance': 0.6388449668884277, 'entity': {'id': 0, 'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}}
{'id': 3, 'distance': 0.31398770213127136, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 2, 'distance': 0.2985413074493408, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
